### Pinecone Vector DB
Create your index and apikey from here https://www.pinecone.io/


In [ ]:
api_key="pcsk_4jq3KF_A4HhT2gy6K1jkyTXtf3VFNpZdPGfxrN8rN3ZkM5fiYa"



In [3]:
pip install -qU langchain langchain-pinecone langchain-openai

In [4]:
from pinecone import Pinecone

pc = Pinecone(api_key=api_key)
pc

In [ ]:
from langchain_openai import OpenAIEmbeddings

# Embedding
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1024,
    api_key="sk-proj-6O_9hQJdgCwarwKoS74yWcKgCRXT3BlbkFJvvyNYgkgAAsbkWIY0hU4vqwsD3CQAuXCDKt8Wu572C653JXZlj-d9U4NZGG1xh-nMUIYguf1kA"
)

embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7c5fb1025700>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7c5faf1096d0>, model='text-embedding-3-small', dimensions=1024, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [6]:
from pinecone import ServerlessSpec

index_name = "pinecone"  # change if desired

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1024,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [7]:
index

In [9]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=index, embedding=embeddings)
vector_store

In [10]:
from langchain_core.documents import Document

# List of sample docs
document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

# List the documents
documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

documents

[Document(metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.'),
 Document(metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic application

In [11]:
# Add to Pinecone vector DB
vector_store.add_documents(documents=documents)

['a9c1cc13-306b-4f0c-ad76-9dc4dbc7f1a3',
 'd4a13559-6c90-40e7-b997-6fa560402b76',
 '7975ebb9-544e-4876-83a3-67f747b551b6',
 '0277bd33-314b-4db1-a535-742dde476de8',
 'f688e6aa-5934-422e-9f67-3a4e50ccd961',
 'd80ffb72-79b2-4825-9b19-0d56668835a5',
 'cca52ebd-65e9-44c4-b0f0-6c40875be24d',
 '2d2a38c7-24a6-433e-a830-33c316ba7e2d',
 '17b76682-b150-48ba-b8fa-44aa9c3fa4cc',
 '7570901f-0e37-4d6c-9d32-9bd42befcbe1']

In [12]:
# Query Directly with similarity serach
query = "What is the weather forecast for tomorrow?"

results = vector_store.similarity_search(
    query,
    k=2,
    filter={"source": "tweet"},
)

for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* I had chocolate chip pancakes and scrambled eggs for breakfast this morning. [{'source': 'tweet'}]
* I have a bad feeling I am going to get deleted :( [{'source': 'tweet'}]


In [19]:
# Similarity search with socre
query = "Will it be hot tomorrow"

results = vector_store.similarity_search_with_score(
    query,
    k=3,
    filter={"source": "news"},
)

for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

* [SIM=0.546278] The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees. [{'source': 'news'}]
* [SIM=0.101219] The stock market is down 500 points today due to fears of a recession. [{'source': 'news'}]
* [SIM=0.028252] Robbers broke into the city bank and stole $1 million in cash. [{'source': 'news'}]


In [21]:
# Retriever
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 1, "score_threshold": 0.4},
)

query = "Stealing from the bank is a crime"

result = retriever.invoke(query, filter={"source": "news"})
result[0].page_content

'Robbers broke into the city bank and stole $1 million in cash.'